# RAG Benchmark: Before vs After HRM Stabilization

This notebook compares a baseline RAG quality proxy against a stabilized pipeline that calls `/v1/stabilize`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [ ]:
BASE_URL = os.getenv('PHASE_LOCK_BASE_URL', 'http://127.0.0.1:8000')
API_KEY = os.getenv('GATEWAY_API_KEY', 'replace-me')
HEADERS = {'Content-Type': 'application/json', 'X-API-Key': API_KEY}

def stabilize_request(tenant_id, request_id, text, embedding, metric_window):
    payload = {
        'tenant_id': tenant_id,
        'request_id': request_id,
        'text': text,
        'embedding': embedding,
        'metric_window': metric_window,
        'metadata': {'source': 'benchmark-notebook', 'step_cap': 120}
    }
    r = requests.post(f'{BASE_URL}/v1/stabilize', headers=HEADERS, json=payload, timeout=20)
    r.raise_for_status()
    return r.json()

In [ ]:
rng = np.random.default_rng(42)
n_queries = 200

rows = []
for i in range(n_queries):
    # Synthetic retrieval-quality regime with drift bursts.
    drift_burst = 0.65 if (i % 50) > 38 else 0.0
    base_retrieval = np.clip(rng.normal(0.72 - drift_burst * 0.35, 0.08), 0.0, 1.0)
    context_noise = np.clip(rng.normal(0.25 + drift_burst * 0.45, 0.10), 0.0, 1.0)

    metric_window = np.clip(rng.normal(loc=base_retrieval - context_noise, scale=0.12, size=48), -1.0, 1.0).tolist()
    embedding = rng.normal(0.0, 0.12, size=24).tolist()

    # Baseline hallucination proxy: more context noise + lower retrieval quality = worse.
    baseline_hallucination = np.clip(0.55 * (1.0 - base_retrieval) + 0.45 * context_noise, 0.0, 1.0)

    out = stabilize_request(
        tenant_id='acme',
        request_id=f'bench-{i:04d}',
        text='Explain the weekly KPI movement using retrieved evidence',
        embedding=embedding,
        metric_window=metric_window,
    )

    # Stabilized hallucination proxy: combine drift score + risk/action policy signal.
    action_penalty = 0.15 if ('fallback-model' in out['actions'] or 'human-review' in out['actions']) else 0.0
    stabilized_hallucination = np.clip(0.75 * out['drift_score'] + 0.25 * baseline_hallucination - action_penalty, 0.0, 1.0)

    rows.append({
        'query_id': i,
        'baseline_hallucination': float(baseline_hallucination),
        'stabilized_hallucination': float(stabilized_hallucination),
        'drift_score': float(out['drift_score']),
        'risk_level': out['risk_level'],
        'actions': ','.join(out['actions']),
        'energy_level': out['diagnostics'].get('energy_level', 0.0),
        'stress_level': out['diagnostics'].get('stress_level', 0.0),
        'topology_volatility': out['diagnostics'].get('topology_volatility', 0.0),
        'surgery_rate': out['diagnostics'].get('surgery_rate', 0.0),
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'avg_baseline_hallucination',
        'avg_stabilized_hallucination',
        'relative_reduction_pct',
        'avg_drift_score',
        'avg_energy_level',
        'avg_stress_level',
        'avg_topology_volatility',
        'avg_surgery_rate'
    ],
    'value': [
        df['baseline_hallucination'].mean(),
        df['stabilized_hallucination'].mean(),
        100.0 * (1.0 - df['stabilized_hallucination'].mean() / df['baseline_hallucination'].mean()),
        df['drift_score'].mean(),
        df['energy_level'].mean(),
        df['stress_level'].mean(),
        df['topology_volatility'].mean(),
        df['surgery_rate'].mean(),
    ]
})
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df['query_id'], df['baseline_hallucination'], label='baseline', alpha=0.8)
axes[0].plot(df['query_id'], df['stabilized_hallucination'], label='stabilized', alpha=0.8)
axes[0].set_title('Hallucination Proxy: Before vs After')
axes[0].set_xlabel('Query')
axes[0].set_ylabel('Proxy score (lower is better)')
axes[0].legend()

axes[1].plot(df['query_id'], df['drift_score'], color='tab:orange')
axes[1].set_title('Drift Score Over Time')
axes[1].set_xlabel('Query')
axes[1].set_ylabel('drift_score')

axes[2].plot(df['query_id'], df['topology_volatility'], label='topology_volatility')
axes[2].plot(df['query_id'], df['surgery_rate'], label='surgery_rate')
axes[2].set_title('Internal HRM Diagnostics')
axes[2].set_xlabel('Query')
axes[2].legend()

plt.tight_layout()
plt.show()